In [1]:
import torch
import numpy as np
from train_featup_test import train_featup, ImageFittingColorFeat
from train_featup_video_test_official_dataloader import VideoAndFlowFitting

# Prepare hr feat video data

In [2]:
'''
    PLEASE READ BEFORE RUNNING THE CODE BELOW
    ------------------------------------------------------------
    to prepare mock data for VideoAndFlowFitting

    run prepare_mock_video.py


    ------------------------------------------------------------
    to prepare real data for VideoAndFlowFitting

    f, c, h, w = video_tensor.shape
    
    data_dict = {
        'motion_type': 'real data',
        'flow_fields': torch.randn(f, 2, h, w), # (f, 2, h, w)
        'video_frames': video_tensor, # (f, c, h, w)
        'source_image': torch.randn(c, h, w), # (c, h, w)
    }

    torch.save(data_dict, data_path)
'''

"\n    to prepare data for VideoAndFlowFitting\n\n    f, c, h, w = video_tensor.shape\n    \n    data_dict = {\n        'motion_type': 'real data',\n        'flow_fields': torch.randn(f, 2, h, w), # (f, 2, h, w)\n        'video_frames': video_tensor, # (f, c, h, w)\n        'source_image': torch.randn(c, h, w), # (c, h, w)\n    }\n\n    torch.save(data_dict, data_path)\n"

In [3]:
WORK_DIR = '/home/tonyz/code_bases/SIREN/siren'

In [ ]:
mockvideo = VideoAndFlowFitting(data_path=f'{WORK_DIR}/data/mock_videos/translation_h.pt')
mockvideo_vid_data_tensor = mockvideo.video_frames.permute(0, 2, 3, 1)

hr_feat_list = []
for frm_idx in range(mockvideo_vid_data_tensor.shape[0]):
    frm = mockvideo_vid_data_tensor[frm_idx, ...].permute(2, 0, 1) # (c, h, w)
    frm = torch.cat([frm for i in range(3)], dim=0)
    print("frm.shape: ", frm.shape)
    hr_feat = train_featup(cameraman=ImageFittingColorFeat(224, img_tensor=frm),
                 do_plot=False)
    print(hr_feat.shape)

    hr_feat_list.append(hr_feat)

mockvideo_hr_feat_2d = torch.stack(hr_feat_list, dim=0)
print(mockvideo_hr_feat_2d.shape)


In [11]:
print(mockvideo_hr_feat_2d.shape)

torch.Size([30, 384, 224, 224])


In [16]:
mockvideo_feat_data = mockvideo_hr_feat_2d.detach().cpu().permute(0, 2, 3, 1).numpy()
np.save(f'{WORK_DIR}/data/mock_videos/mockvideo_feat_data.npy', 
        mockvideo_feat_data)
